# The baseline closed loop, step by step

One real shot through the simplest loop: syndrome data in -> controller (pulses to binary) -> syndrome packing -> Buffer 0 -> window manager -> weak decoder (its own memory; fetch, algorithm, release at a clock) -> Pauli frame. No reorder buffer, no strong tier, no confidence estimator.

Every number shown is measured inside the simulator on this run. The configured costs all come from one file, `experiments/baseline/baseline_closed_loop.yaml`.


In [1]:
import os
import sys

notebook_directory = os.getcwd()
repo_from_notebook_directory = os.path.join(notebook_directory, "..", "..")
running_inside_baseline_folder = notebook_directory.endswith("baseline")
if running_inside_baseline_folder:
    REPO = os.path.abspath(repo_from_notebook_directory)
else:
    REPO = notebook_directory
sys.path.insert(0, REPO)
os.chdir(REPO)

from decsim.config import microseconds


def row_as_strings(row: dict, columns: list) -> list:
    """One table row: the row's value under each column, as text."""
    values = []
    for column in columns:
        value = row.get(column, "")
        values.append(str(value))
    return values


def column_widths(columns: list, cells: list) -> list:
    """Width of each column: its header or its widest cell."""
    widths = []
    for index, column in enumerate(columns):
        width = len(column)
        for row in cells:
            width = max(width, len(row[index]))
        widths.append(width)
    return widths


def aligned_line(values: list, widths: list) -> str:
    """One printed line: each value padded to its column width."""
    padded = []
    for value, width in zip(values, widths):
        padded.append(value.ljust(width))
    return "  ".join(padded)


def table(rows: list, columns: list) -> None:
    """Print rows (dicts) as an aligned text table with the given columns."""
    cells = []
    for row in rows:
        cells.append(row_as_strings(row, columns))
    widths = column_widths(columns, cells)
    rules = []
    for width in widths:
        rules.append("-" * width)
    print(aligned_line(columns, widths))
    print(aligned_line(rules, widths))
    for row in cells:
        print(aligned_line(row, widths))


print("repo:", REPO)


repo: /scratch/gpfs/MARTONOSI/sk2415/qlx-qec-sandbox/decsim


## 1. The configuration: every cost in one place

Round period, decoder card, engine cycles and clock, controller times, the link cards, the Pauli frame commit. Nothing else in the loop carries a number.


In [2]:
CONFIG_PATH = "experiments/baseline/baseline_closed_loop.yaml"
print(open(CONFIG_PATH).read())


# Baseline closed loop: the single source of every parameter of the experiment.
# Every number that costs simulated time carries its source next to it.

code_task: surface_code:rotated_memory_z     # stim generator task (real detector data)
distance: 3
rounds_per_shot: 60                          # QEC rounds per shot; windows are (commit d, buffer d) sliding
noise_probability: 0.001                     # all four stim noise channels
seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]        # one shot per seed at each sweep point

# Sweep axis 1: input round period (the QEC cycle time seen by the controller).
round_period_us: [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
# Sweep axis 2: weak decoder algorithm latency (one window), microseconds, or "measured":
# 0.028 = LILLIPUT [d=3, m=2] 7 cycles at 250 MHz, 2108.06569 sec. 6.3 and Table 4 (ASIC card).
# measured = wall clock of each real PyMatching call on this host (software decoder row).
algorithm_latency_us: [0.028, 0.28, measured]

controller:
  t_binary_

## 2. Tracing hooks

Each stage of the loop is a real method. We wrap those methods so every call leaves one record in `TRACE` (simulated time, stage name, the data that passed). The loop itself is untouched.


In [3]:
from decsim.qpu.cycle_clock import QPUDevice
from decsim.controller.controller import Controller
from decsim.controller.syndrome_ingress import SyndromeIngress
from decsim.syndrome_buffer.syndrome_buffer import SyndromeBuffer
from decsim.windows.window_manager import WindowManager
from decsim.windows.window_boundaries import BoundaryCourier
from decsim.decoders.decoder_manager import DecoderManager
from decsim.decoders.decoder_memory import DecoderMemory
from decsim.pauli_frame.pauli_frame import PauliFrame

TRACE = []
CLOCK = {"engine": None}


def record(stage: str, **data) -> None:
    """Append one trace record stamped with the simulated time."""
    now_us = microseconds(CLOCK["engine"].now)
    TRACE.append(dict(t=now_us, stage=stage, **data))


def wrap(cls, method_name: str, before=None, after=None) -> None:
    """Replace cls.method_name with a version that calls before(...) first and after(...) last."""
    original = getattr(cls, method_name)

    def wrapped(self, *args, **kwargs):
        if before is not None:
            before(self, *args, **kwargs)
        result = original(self, *args, **kwargs)
        if after is not None:
            after(self, result, *args, **kwargs)
        return result

    setattr(cls, method_name, wrapped)


def bits_as_text(bits) -> str:
    text = ""
    for bit in bits:
        text += str(int(bit))
    return text


# ---- stage hooks, in path order

def on_qpu_issue(qpu, command):
    CLOCK["engine"] = qpu.engine
    record("controller -> qpu: issue", op=command.operation.name, rounds=command.round_count,
           cycle_us=microseconds(command.round_ticks), starts_at=microseconds(qpu.next_boundary()))


def on_qpu_round(qpu, payloads, operation):
    for payload in payloads:
        record("qpu: round emitted", op=operation.name, round=payload.round_index,
               bits=bits_as_text(payload.bits), size_bits=payload.size_bits)


def on_controller_readout(controller, readout, route):
    binary_us = microseconds(controller.binary_availability_ticks)
    record("controller: readout accepted (pulses -> binary)", round=readout.round_index, binary_us=binary_us)


def on_packing(ingress, payload, route, **kwargs):
    fragment = f"{payload.fragment_index + 1}/{payload.n_fragments}"
    record("packing: fragment relayed", round=payload.round_index, fragment=fragment, size_bits=payload.size_bits)


def on_buffer_retained(buffer, packet, round_identity, **kwargs):
    snapshot = buffer.snapshot()
    retained_rounds = []
    for identity in snapshot.retained_identities:
        retained_rounds.append(identity[1])
    first_fragment = packet.fragments[0]
    record("buffer 0: round retained", round=round_identity[1], bits=bits_as_text(first_fragment.bits),
           occupancy=snapshot.occupancy, retained=retained_rounds)


def window_state_text(window_manager) -> str:
    parts = []
    for (op_id, window_index), window in sorted(window_manager.windows.items()):
        state = "READY" if window.t_data_complete else "waiting"
        parts.append(f"W{window_index}[{window.commit_lo}-{window.commit_hi}|buf->{window.buffer_hi}] {state}")
    return "  ".join(parts)


def on_window_round(window_manager, result, packet):
    record("window manager: round arrived", round=packet.round_index, windows=window_state_text(window_manager))


def free_units_of(decoder_manager) -> dict:
    free = {}
    for pool, units in decoder_manager._free_units.items():
        free[pool] = list(units)
    return free


def on_enqueue(decoder_manager, job, reserve_transfer=None):
    record("decoder manager: window enqueued", window=job.label, free_units=free_units_of(decoder_manager))


def on_unit_assigned(decoder_manager, result, pool, job):
    record("decoder manager: unit assigned", window=job.label, unit=job.unit, free_units=free_units_of(decoder_manager))


def on_memory_deposit(memory, decoder_input, job):
    rounds = []
    for round_packet in decoder_input.rounds:
        rounds.append(round_packet.round_index)
    record("decoder memory: input landed in unit", unit=memory.unit, window=job.label, rounds=rounds,
           occupied_rounds=memory.occupied_rounds)


def on_decode_start(decoder_manager, job):
    record("decoder manager: start decode", window=job.label, unit=job.unit)


def on_decode_result(decoder_manager, job, result):
    rounds_fetched = []
    defects = 0
    for fragment in job.payloads:
        rounds_fetched.append(fragment.round_index)
        defects += int(sum(fragment.bits))
    correction_weight = None
    if result.correction is not None:
        correction_weight = int(sum(result.correction))
    record("decoder engine: result", window=job.label, syndrome_rounds_fetched=rounds_fetched, defects=defects,
           correction_weight=correction_weight, logical=result.logical_observables, boundary_defects=result.boundary_defects)


def on_boundary_sent(courier, window, operation, boundary, **kwargs):
    dependents = []
    for (_, window_index) in window.dependents:
        dependents.append(f"W{window_index}")
    record("boundary handoff (DD) at decode done", window=f"W{window.k}", to=dependents, boundary=boundary)


def on_frame_commit(frame, **kwargs):
    window_index = kwargs["window_key"][1]
    record("pauli frame: correction arrived (after WDO)", window=f"W{window_index}", logical=kwargs["logical_observables"])


wrap(QPUDevice, "issue", before=on_qpu_issue)
wrap(QPUDevice, "_emit", before=on_qpu_round)
wrap(Controller, "accept_qpu_readout", before=on_controller_readout)
wrap(SyndromeIngress, "relay_qpu_readout", before=on_packing)
wrap(SyndromeBuffer, "finish_packing", after=on_buffer_retained)
wrap(WindowManager, "on_syndrome_arrival", after=on_window_round)
wrap(DecoderManager, "enqueue", before=on_enqueue)
wrap(DecoderManager, "_start_job", after=on_unit_assigned)
wrap(DecoderMemory, "deposit", after=on_memory_deposit)
wrap(DecoderManager, "_begin_service", before=on_decode_start)
wrap(DecoderManager, "_on_decode_done", before=on_decode_result)
wrap(BoundaryCourier, "send", before=on_boundary_sent)
wrap(PauliFrame, "commit_weak_correction", before=on_frame_commit)
print("hooks installed:", 13)


hooks installed: 13


## 3. One traced shot

The run is built from the yaml with three changes so the walkthrough fits on screen: 9 rounds instead of 60, noise 0.02 instead of 0.001 (so defects and corrections are visible), and the weak decoder as real PyMatching whose algorithm stage lasts the wall clock of the call. Round period 1.0 us, seed 0.


In [4]:
from experiments.baseline.baseline_closed_loop import build_run, load_config

config = load_config(CONFIG_PATH)
config["rounds_per_shot"] = 9
config["noise_probability"] = 0.02

spec, decoder_engine = build_run(config, round_period_us=1.0, algorithm_latency_us="measured", seed=0)
done = spec.build()

print("terminal status:", done.result.terminal_status)
print("events traced:", len(TRACE))


terminal status: complete
events traced: 60
